# PyBullet 的 EGL 渲染器：快 4.4 倍，但会把 mask 毁掉

<a href="https://colab.research.google.com/github/yangyi02/droid/blob/main/notebooks/pybullet_egl_mask_benchmark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

`core/physics.py` 里的 `PyBulletRenderer` 一直跑在 CPU 光栅化器上，而且不是有意选的——
它想加载 GPU（EGL）插件，只是查错了模块名，于是十几个月里每一帧都悄悄走了 CPU。

把名字改对，确实快 4.4 倍。但**机器人 mask 会塌到原来的 1/12**，而 mask 正是
`compute_tracks` 的输入。这个 notebook 把这两件事都测给你看，然后给出一个
速度和正确性都要的改法。

结论预告（A100，1280×720，Franka 手臂 + Robotiq 夹爪双 body 场景）：

| 配置 | 每帧 | mask 覆盖 | 与 CPU 的 mask IoU |
|---|---|---|---|
| CPU 光栅化器（今天的行为） | 124 ms | 4.425% | — |
| EGL + `alpha=0` 隐藏（只把插件名改对） | 28 ms | **0.357%** | **0.08** |
| EGL + 从 URDF 里删掉几何（本文方案） | 28 ms | 4.439% | **0.98** |

下面每个数字都是这个 notebook 自己跑出来的，不是抄的。

---

### 跑之前请先看这三条

**1. 需要 GPU runtime。** Colab 上：`代码执行程序 → 更改运行时类型 → T4 GPU`。
没有 GPU 就加载不了 EGL 插件，所有 EGL 的格子会静默退回 CPU，对照就不成立了。
第 0 节会明确告诉你有没有加载上。

**2. 会编译一次 PyBullet。** 96 核约 1 分钟，Colab 两核十几分钟。
为什么必须自己编见 [pybullet_numpy_benchmark.ipynb](pybullet_numpy_benchmark.ipynb)：
没有 NumPy 支持的话，每帧多出来的 ~300 ms tuple 编组会把 CPU/GPU 的差距整个淹掉。
已经装好的话这一步跳过。

**3. 会在两个 URDF 旁边写临时文件**（`_trimmed_*.urdf`），最后一节负责删掉。

## 0. 环境

In [ ]:
# @title 0a. 找到仓库
# Same as the other notebooks here: clone on Colab, and locally walk up from
# cwd rather than assuming it -- this file lives in notebooks/.
import os
import subprocess
import sys

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_DIR = "/content/droid"
    if not os.path.exists(REPO_DIR):
        subprocess.run(["git", "clone", "--recursive",
                        "https://github.com/yangyi02/droid.git", REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    REPO_DIR = os.getcwd()
    while not os.path.exists(os.path.join(REPO_DIR, "core", "physics.py")):
        parent = os.path.dirname(REPO_DIR)
        if parent == REPO_DIR:
            REPO_DIR = os.getcwd()
            break
        REPO_DIR = parent

if not os.path.exists(os.path.join(REPO_DIR, "core", "physics.py")):
    raise SystemExit(f"{REPO_DIR} is not the droid repo root -- open this "
                     "notebook from the checkout, or set REPO_DIR by hand.")
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print(f"{'Colab' if IN_COLAB else 'Local'}: {REPO_DIR}")

In [ ]:
# @title 0b. PyBullet，必须是带 NumPy 支持的构建
# Without it getCameraImage marshals every pixel into a Python tuple, which
# costs ~300 ms a frame on top of *both* rasterisers and would hide the very
# difference this notebook is about. See pybullet_numpy_benchmark.ipynb.
import subprocess
import sys

def numpy_enabled():
    out = subprocess.run([sys.executable, "-c",
                          "import pybullet as p; print(p.isNumpyEnabled())"],
                         capture_output=True, text=True)
    return out.stdout.strip().endswith("1")

if numpy_enabled():
    print("pybullet already has NumPy support -- nothing to build")
else:
    print("Rebuilding pybullet from source. Minutes. Output streams below.")
    cmd = [sys.executable, "-m", "pip", "install", "--force-reinstall",
           "--no-deps", "--no-binary", "pybullet", "--no-build-isolation",
           "--no-cache-dir", "pybullet"]
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        if any(k in line for k in ("numpy is", "Building wheel", "Created wheel",
                                   "Successfully", "error")):
            print("   ", line.rstrip())
    print("   isNumpyEnabled =", numpy_enabled())
    if IN_COLAB:
        print("\n>>> Colab 需要重启运行时才能换掉已经导入的 .so："
              "\n>>> 代码执行程序 → 重新启动会话，然后从这里继续。")

In [ ]:
# @title 0c. 有没有 GPU
# The EGL plugin is the whole point of the comparison, so fail loudly rather
# than silently measuring CPU twice.
import importlib.util

import pybullet as p

assert p.isNumpyEnabled(), "上一格装完了吗？Colab 上可能需要先重启运行时"

p.connect(p.DIRECT)
spec = importlib.util.find_spec("eglRenderer")
EGL_OK = spec is not None and p.loadPlugin(spec.origin, "_eglRendererPlugin") >= 0
p.disconnect()

print("eglRenderer module :", spec.origin if spec else "NOT FOUND")
print("EGL plugin loads   :", EGL_OK)
if not EGL_OK:
    print("\n没有可用的 GPU 光栅化器。后面的 EGL 结果会退回 CPU，对照不成立。"
          "\nColab: 代码执行程序 → 更改运行时类型 → T4 GPU，然后从头再跑。")

## 1. 出发点：那个插件从来没被加载过

`core/physics.py` 在 2026-09-02 之前是这么写的：

```python
egl_spec = importlib.util.find_spec('eglRendererPlugin')
if egl_spec:
    p.loadPlugin(egl_spec.origin, "_eglRendererPlugin")
```

问题在第一行查的名字。pybullet 装出来的**模块**叫 `eglRenderer`；
`_eglRendererPlugin` 只是它在 pybullet 内部**注册时用的名字**，不是模块名。
所以 `find_spec` 永远返回 `None`，`if` 永远不成立，插件永远不加载——
而 `ER_BULLET_HARDWARE_OPENGL` 在没有插件时会**静默退回** CPU 的 TinyRenderer。

没有报错，没有警告，只是慢 4.5 倍。

In [ ]:
import importlib.util

for name in ("eglRendererPlugin", "eglRenderer"):
    spec = importlib.util.find_spec(name)
    print(f"find_spec({name!r:22}) -> {spec.origin if spec else None}")

## 2. 三种配置

只改名字（下面的 `hide='alpha'`）不是终点，因为 `PyBulletRenderer` 的整个设计
建立在一个 CPU 光栅化器才认的约定上。

它同时加载**两个 body**：一个 Franka 手臂（把 hand / finger 隐藏），一个 Robotiq
夹爪（把整条 `panda_link*` 手臂隐藏），两个拼起来才是完整的机器人。而"隐藏"用的是
`changeVisualShape(..., rgbaColor=[0, 0, 0, 0])`——把 alpha 设成 0。

所以要测三种：

| `hide=` | 隐藏方式 | 是什么 |
|---|---|---|
| `'alpha'` | 原样保留 `alpha=0` | 只把插件名改对的"一行修复" |
| `'visual'` | 从 URDF 删掉 `<visual>` | 看起来该行，但有个坑 |
| `'geometry'` | 删掉 `<visual>` **和** `<collision>` | 真正能用的方案 |

In [ ]:
import hashlib
import importlib.util
import os
import time
import xml.etree.ElementTree as ET

import numpy as np
import pybullet as p
import pybullet_data

from core.physics import PyBulletRenderer


def load_egl():
    """Load the GPU rasteriser plugin into the *current* connection.

    Two traps: the module pybullet ships is `eglRenderer` (`_eglRendererPlugin`
    is only the name it registers under), and the plugin only sees geometry
    registered after it loads -- call this before loadURDF or you will render,
    and benchmark, an empty scene.
    """
    spec = importlib.util.find_spec('eglRenderer')
    return spec is not None and p.loadPlugin(spec.origin, '_eglRendererPlugin') >= 0


def trim_urdf(src, hide, drop=('visual', 'collision')):
    """Copy `src`, stripping `drop` tags from links whose name matches `hide`.

    Written beside the original so relative and package:// mesh paths resolve
    exactly as before.
    """
    tree = ET.parse(src)
    stripped = []
    for link in tree.getroot().findall('link'):
        if hide(link.get('name', '')):
            gone = [e for tag in drop for e in link.findall(tag)]
            for e in gone:
                link.remove(e)
            if gone:
                stripped.append(link.get('name'))
    tag = hashlib.md5((src + repr(drop) + repr(stripped)).encode()).hexdigest()[:8]
    out = os.path.join(os.path.dirname(src), f'_trimmed_{tag}.urdf')
    tree.write(out)
    return out, stripped


# Which links each body wants hidden -- the same rule core/physics.py applies
# with alpha=0: the arm body hides its hand and fingers, the ghost body hides
# the whole arm, and together they make one complete robot.
HIDE = {'robot': lambda n: 'hand' in n or 'finger' in n,
        'ghost': lambda n: 'panda_link' in n}


class EGLRenderer(PyBulletRenderer):
    """PyBulletRenderer on the GPU rasteriser, with a choice of how to hide links.

    hide='alpha'    the parent's rgbaColor alpha=0, untouched -- the naive fix
    hide='visual'   drop <visual> from the hidden links
    hide='geometry' drop <visual> *and* <collision> -- the one that works
    """

    def __init__(self, hide='geometry', ghost_urdf=None):
        self.hide, self.egl, self.stripped = hide, False, {}
        real_load, real_vis = p.loadURDF, p.changeVisualShape

        def load(path, *a, **kw):
            if not self.egl:                       # before the first body
                self.egl = load_egl()
            if self.hide != 'alpha':
                which = 'robot' if path.endswith('panda.urdf') else 'ghost'
                src = path if os.path.isabs(path) else os.path.join(
                    pybullet_data.getDataPath(), path)
                drop = (('visual',) if self.hide == 'visual'
                        else ('visual', 'collision'))
                path, self.stripped[which] = trim_urdf(src, HIDE[which], drop)
            return real_load(path, *a, **kw)

        p.loadURDF = load
        if self.hide != 'alpha':
            p.changeVisualShape = lambda *a, **kw: None   # nothing left to hide
        try:
            super().__init__(ghost_urdf)
        finally:
            p.loadURDF, p.changeVisualShape = real_load, real_vis

    def _render_raw(self, extrinsic, K, w, h):
        cam = extrinsic[:3, 3]
        view = p.computeViewMatrix(cam.tolist(), (cam + extrinsic[:3, 2]).tolist(),
                                   (-extrinsic[:3, 1]).tolist())
        _, _, _, depth, seg = p.getCameraImage(
            w, h, viewMatrix=view,
            projectionMatrix=self._get_projection_matrix(K, w, h),
            renderer=p.ER_BULLET_HARDWARE_OPENGL,
            flags=p.ER_SEGMENTATION_MASK_OBJECT_AND_LINKINDEX)
        return depth, seg

In [ ]:
W, H = 1280, 720
CAM = np.array([1.2, 0.0, 0.6])
TARGET = np.array([0.0, 0.0, 0.3])
K = np.array([[640., 0., W / 2], [0., 640., H / 2], [0., 0., 1.]])


def camera_pose(cam=CAM, target=TARGET):
    """4x4 extrinsic in the convention _render_raw expects (+z forward, +y down)."""
    z = target - cam
    z = z / np.linalg.norm(z)
    x = np.cross(z, [0, 0, 1.0])
    x = x / np.linalg.norm(x)
    ext = np.eye(4)
    ext[:3, :3] = np.stack([x, np.cross(z, x), z], axis=1)
    ext[:3, 3] = cam
    return ext


EXT = camera_pose()


def timeit(fn, warmup=8, n=25):
    """Median ms. The warmup matters: a cold GPU context reads ~40% slow."""
    for _ in range(warmup):
        fn()
    ts = []
    for _ in range(n):
        t0 = time.perf_counter()
        fn()
        ts.append((time.perf_counter() - t0) * 1000)
    return float(np.median(ts))


def measure(renderer):
    """Everything one configuration has to say, in one pass.

    Also keeps the raw segmentation buffer and the link bookkeeping, because
    each renderer disconnects the previous one -- whatever is not captured here
    cannot be asked for later.
    """
    depth = renderer.render_depth(EXT, K, W, H)
    mask = renderer.render_mask(EXT, K, W, H)
    if (depth > 0).sum() < 1000:
        raise RuntimeError('empty render -- timing it would be meaningless')
    _, seg = renderer._render_raw(EXT, K, W, H)
    seg = np.reshape(seg, (H, W)).astype(np.int32)
    return dict(depth=depth, mask=mask, seg=seg,
                obj=seg & 0xFFFFFF, link=(seg >> 24) - 1,
                ids={'robot': renderer.robot_id, 'ghost': renderer.ghost_id},
                hidden={'robot': renderer.hidden_robot_links,
                        'ghost': renderer.hidden_ghost_links},
                ms=timeit(lambda: renderer.render_depth(EXT, K, W, H)))


def links_drawn(r, body):
    """Which links of  actually reached the screen, and which of those
    were supposed to be hidden."""
    drawn = sorted(np.unique(r['link'][r['obj'] == r['ids'][body]]).tolist())
    return drawn, [l for l in drawn if l in r['hidden'][body]]

## 3. 跑

四个配置依次构造。`PyBulletRenderer.__init__` 每次都会 `disconnect()` 再
`connect()`，所以它们互不干扰——但也因此，**上一个渲染器的任何东西事后都问不到了**，
`measure()` 里一次性把要用的都留下来。

In [ ]:
import warnings

warnings.simplefilter("ignore")   # the URDF importer is chatty about inertials

RES = {"cpu": measure(PyBulletRenderer())}
for how in ("alpha", "visual", "geometry"):
    RES[how] = measure(EGLRenderer(hide=how))

print(f"\n{'配置':<12}{'每帧':>10}{'depth 覆盖':>14}{'mask 覆盖':>13}")
for k, r in RES.items():
    print(f"{k:<12}{r['ms']:>8.1f}ms{100 * (r['depth'] > 0).mean():>12.3f}%"
          f"{100 * r['mask'].mean():>12.3f}%")

## 4. 证据一：mask 塌了

`depth 覆盖` 三种 EGL 配置都在 4.4% 上下——**深度图看起来没事**。
`mask 覆盖` 才是出事的地方：`alpha` 从 4.425% 掉到 0.357%，少了 12 倍。

In [ ]:
import numpy as np

base = RES["cpu"]["mask"]
print(f"{'配置':<12}{'mask 覆盖':>12}{'与 CPU 的 IoU':>16}")
for k, r in RES.items():
    iou = (base & r["mask"]).sum() / (base | r["mask"]).sum()
    print(f"{k:<12}{100 * r['mask'].mean():>11.3f}%{iou:>16.4f}")

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import numpy as np

# Categorical slots 1-3 of the reference palette; the trio validates all-pairs
# (worst CVD dE 9.2, worst normal-vision dE 24.0). Figure text stays ASCII --
# matplotlib's default font ships no CJK glyphs.
C_CPU, C_BAD, C_FIX = "#2a78d6", "#eb6834", "#1baf7a"
SURFACE, INK, MUTED, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e5e4e0"
PANELS = [("CPU rasteriser\n(what ships today)", "cpu", C_CPU),
          ("EGL, alpha=0 hiding\n(the one-line fix)", "alpha", C_BAD),
          ("EGL, geometry removed\n(the fix that works)", "geometry", C_FIX)]

ys, xs = np.where(np.logical_or.reduce([RES[k]["mask"] for _, k, _ in PANELS]))
box = (slice(max(ys.min() - 24, 0), ys.max() + 24),
       slice(max(xs.min() - 24, 0), xs.max() + 24))

fig, axes = plt.subplots(1, 3, figsize=(9.4, 3.8), facecolor=SURFACE)
for ax, (label, key, colour) in zip(axes, PANELS):
    m = RES[key]["mask"][box]
    rgb = np.ones((*m.shape, 3))
    rgb[m] = matplotlib.colors.to_rgb(colour)
    ax.imshow(rgb, interpolation="nearest")
    ax.set_title(label, fontsize=9.5, color=INK, pad=8)
    ax.set_xlabel("%.3f%% of frame" % (100 * RES[key]["mask"].mean()),
                  fontsize=9, color=MUTED)
    ax.set_xticks([])
    ax.set_yticks([])
    for s in ax.spines.values():
        s.set_color(GRID)
fig.suptitle("Robot mask, same pose and camera", fontsize=11, color=INK, y=1.0)
fig.tight_layout()
plt.show()

中间那张就是全部的问题：本该是一整条手臂，只剩夹爪的两根手指。

## 5. 证据二：为什么 —— EGL 不理会 `alpha=0`

分割缓冲区里每个像素编了 `objectId | (linkIndex + 1) << 24`，所以可以直接问：
**哪些 link 真的画到屏幕上了，其中哪些是本该被隐藏的？**

In [ ]:
for k in ("cpu", "alpha"):
    print(k)
    for body in ("robot", "ghost"):
        drawn, leaked = links_drawn(RES[k], body)
        print(f"  {body:<6} 画出来的 link: {drawn}")
        print(f"  {'':<6} 其中本该隐藏: {leaked}")

CPU 那边两行 `本该隐藏` 都是空的——`alpha=0` 生效了。

EGL 那边，手臂 body 把 hand/finger（link 8、9、10）照画不误，夹爪 body 把整条
`panda_link*` 手臂也照画不误。于是**夹爪那条本该隐形的手臂，挡在了真手臂前面**。
深度图看不出来（两条手臂几乎重合，深度差不多），但分割 id 是夹爪的，
`render_mask` 按 link id 一过滤，真手臂就整条消失了。

结论：`alpha=0` 是个只有 TinyRenderer 认的约定。要换光栅化器，得先换隐藏方式。

## 6. 方案：从 URDF 里把几何删掉

不再依赖渲染器怎么解释 alpha，直接让那些 link **没有可画的东西**——
复制一份 URDF，把要隐藏的 link 的几何标签删掉，加载这份副本。
临时文件写在原文件旁边，这样 `meshes/...` 和 `package://...` 的相对路径解析方式完全不变。

这里有个坑，值得单独看一眼：**只删 `<visual>` 是不够的**。
一个 link 如果没有 visual，pybullet 会拿它的 `<collision>` 几何来渲染——
形状更粗、还略胖一圈。这正是上面 `hide='visual'` 那一行的作用：

In [ ]:
import numpy as np

base_d = RES["cpu"]["depth"]
print(f"{'配置':<12}{'depth 覆盖':>12}{'与 CPU 深度中位差':>20}{'超 1.5cm 阈值':>16}")
for k in ("visual", "geometry"):
    d = RES[k]["depth"]
    both = (base_d > 0) & (d > 0)
    diff = np.abs(base_d - d)[both]
    print(f"{k:<12}{100 * (d > 0).mean():>11.3f}%{np.median(diff):>18.2e} m"
          f"{100 * (diff > 1.5e-2).mean():>15.2f}%")

`visual`：覆盖率从 4.4% 涨到 6.9%（多出来的就是那圈碰撞体），
深度中位差 3.6 cm，90% 以上的像素超过流水线 1.5e-2 m 的阈值。删一半等于没删。

`geometry`：两个标签一起删，才真的干净。

## 7. 修好之后：等价性和速度

`geometry` 和今天的 CPU 输出比：

In [ ]:
import numpy as np

base_m, base_d = RES["cpu"]["mask"], RES["cpu"]["depth"]
fix = RES["geometry"]

iou = (base_m & fix["mask"]).sum() / (base_m | fix["mask"]).sum()
both = (base_d > 0) & (fix["depth"] > 0)
diff = np.abs(base_d - fix["depth"])[both]

print(f"mask IoU            : {iou:.4f}   (alpha 那版是 "
      f"{(base_m & RES['alpha']['mask']).sum() / (base_m | RES['alpha']['mask']).sum():.4f})")
print(f"两边都算机器人的像素 : {both.sum()}")
for q in (50, 90, 99, 99.9, 100):
    print(f"  深度差 p{q:<5}     : {np.percentile(diff, q):.3e} m")
print(f"超过 1.5e-2 m 阈值   : {100 * (diff > 1.5e-2).mean():.2f}% 的机器人像素")
print(f"\n每帧              : {RES['cpu']['ms']:.1f} ms -> {fix['ms']:.1f} ms"
      f"  ({RES['cpu']['ms'] / fix['ms']:.1f}x)")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(10.5, 2.9), facecolor=SURFACE)
labels = [l.split("\n")[0] for l, _, _ in PANELS]
colours = [c for _, _, c in PANELS]
base_m = RES["cpu"]["mask"]
series = [("ms per 1280x720 frame  (lower is better)",
           [RES[k]["ms"] for _, k, _ in PANELS], "%.0f ms"),
          ("mask IoU vs the CPU baseline  (1.0 is identical)",
           [(base_m & RES[k]["mask"]).sum() / (base_m | RES[k]["mask"]).sum()
            for _, k, _ in PANELS], "%.2f")]

for ax, (title, vals, fmt) in zip(axes, series):
    y = np.arange(3)[::-1]
    ax.barh(y, vals, height=0.34, color=colours, zorder=3)
    for yi, v in zip(y, vals):
        ax.text(v + max(vals) * 0.025, yi, fmt % v, va="center",
                fontsize=9.5, color=INK)
    ax.set_yticks(y)
    ax.set_yticklabels(labels, fontsize=9, color=MUTED)
    ax.set_title(title, fontsize=9.5, color=INK, loc="left", pad=10)
    ax.set_xlim(0, max(vals) * 1.22)
    ax.set_facecolor(SURFACE)
    ax.xaxis.set_visible(False)
    for side in ("top", "right", "bottom"):
        ax.spines[side].set_visible(False)
    ax.spines["left"].set_color(GRID)
    ax.tick_params(length=0)
fig.tight_layout()
plt.show()

## 8. 那 1.7% 和 0.7% 到底在哪

`geometry` 和 CPU 不是逐位相同的，所以问题不是"一样吗"，而是"差在哪"。
mask 的分歧和 depth 的分歧位置不同，得分开量。

判据用局部深度跳变：以每个像素 3x3 邻域内的深度极差衡量,
在光滑表面上它接近 0，在 link 交界和自遮挡边界上它等于两个面的间距。

In [ ]:
from scipy import ndimage

cpu, fix = RES["cpu"], RES["geometry"]
mc, mg = cpu["mask"], fix["mask"]
dc, dg = cpu["depth"], fix["depth"]
both = (dc > 0) & (dg > 0)
diff = np.abs(dc - dg)

# distance to the CPU mask's own silhouette
edge = mc ^ ndimage.binary_erosion(mc)
dist = ndimage.distance_transform_edt(~edge)

disagree = mc ^ mg
print(f"mask 不一致像素 : {disagree.sum()} / {mc.sum()} = "
      f"{100 * disagree.sum() / mc.sum():.2f}%")
for r in (1, 2):
    print(f"   距轮廓 <={r}px : {100 * (dist[disagree] <= r).mean():.1f}%")

# local 3x3 depth range: ~0 on a smooth surface, large across a depth step
big = both & (diff > 1.5e-2)
hi = ndimage.maximum_filter(np.where(dc > 0, dc, -1e9), size=3)
lo = ndimage.minimum_filter(np.where(dc > 0, dc, +1e9), size=3)
jump = hi - lo

print(f"\ndepth 超 1.5e-2 m : {big.sum()} px ({100 * big.sum() / both.sum():.2f}% of shared)")
print(f"   距轮廓 <=1px           : {100 * (dist[big] <= 1).mean():.1f}%")
print(f"   处在深度跳变 >1e-2 m 处 : {100 * (jump[big] > 1e-2).mean():.1f}%"
      f"   (全部机器人像素里只有 {100 * (jump[both] > 1e-2).mean():.1f}% 这样)")

smooth = both & (jump < 1e-3)
print(f"\n只看光滑表面像素 ({smooth.sum()} px, {100 * smooth.sum() / both.sum():.1f}%):")
print(f"   中位差 {np.median(diff[smooth]):.2e} m | 最大 {diff[smooth].max():.2e} m"
      f" | 超阈值 {(diff[smooth] > 1.5e-2).sum()} 个")

## 结论

1. `core/physics.py` 查错了模块名，EGL 插件从来没加载过，一直在跑 CPU 光栅化器。
2. 只把名字改对：快 4.4 倍，但 mask IoU 掉到 0.08 —— 因为 EGL 不认 `alpha=0`，
   夹爪 body 那条本该隐形的手臂挡住了真手臂。**这个改法会静默污染 `compute_tracks` 的输入。**
3. 改成从 URDF 里删掉几何（`<visual>` 和 `<collision>` 都要删），
   速度一样快，mask IoU 0.98。

### 还没解决的那 0.7%

即使用 `geometry`，仍有约 0.7% 的机器人像素深度差超过 1.5e-2 m，
mask 也有 1.7% 的像素不一致。两者的位置不一样，值得分开说：

- **mask 的 1.7%**：99.7% 在轮廓 1px 以内，100% 在 2px 以内。纯粹是两种光栅化器
  对"边界这个像素算不算被覆盖"判定不同。
- **depth 的 0.7%**：只有 9.3% 在外轮廓附近，99.3% 落在**深度不连续处**——
  link 交界、手臂自遮挡的边界。那里差的不是深度算错，而是这个像素归前面还是后面那个面，
  一旦判定不同，差值就等于两个面之间的间距。
- **真正的表面上**（局部 3x3 深度跳变 < 1 mm 的像素，占 14%）：
  中位差 1.6e-4 m，**最大 6.9e-4 m，超阈值的一个都没有**。

所以两条路径在"面"上是亚毫米一致的，分歧全在"边"上。落地前应该在真实 episode 上
量一遍这些边界像素对下游指标的实际影响，而不是只看这一个 pose。

### 怎么落地到 `core/physics.py`

需要三处改动，都不大：

1. `__init__` 里在 **第一次 `loadURDF` 之前** 加载 EGL 插件（顺序是硬性的，
   插件只能看到它加载之后注册的几何——先加载 body 再加载插件会渲染出一片空白，
   而且那个空白还跑得飞快，很容易被当成"提速成功"）。
2. 两个 URDF 都换成删过几何的副本，`changeVisualShape` 那套 alpha 隐藏可以整个去掉。
   `hidden_robot_links` / `hidden_ghost_links` 两个列表要保留，`render_mask` 还在用。
3. `_render_raw` 里把 `renderer=` 换回 `p.ER_BULLET_HARDWARE_OPENGL`，
   并且在插件没加载上时明确退回 `ER_TINY_RENDERER`，不要静默。

副本文件放哪需要想一下：它必须和原 URDF 同目录（mesh 相对路径），
而其中一个原 URDF 在 `assets/` 里，是 git 跟踪的——生成物应该带上 `.gitignore`，
或者改成在 `data/cache/` 下重建一份完整的 `assets/` 副本。

In [ ]:
# @title 清理临时 URDF
import glob
import os

import pybullet_data

removed = 0
for d in (os.path.join(pybullet_data.getDataPath(), "franka_panda"),
          os.path.join(REPO_DIR, "assets", "franka_description")):
    for f in glob.glob(os.path.join(d, "_trimmed_*.urdf")):
        os.remove(f)
        removed += 1
print(f"removed {removed} temporary URDF(s)")